# Domain Data Embedding Pipeline
This notebook sends text data to a Google Cloud Run function that calls the Gemini text embedding service and saves the results to a CSV file.

Textual data that is embeded encodes their semantic info into vectors that can be analyzed by later ML.

This script is designed for the generic "INPUT_FILE" config to be changed for whatever set of strings needs to be embedded.

Authentication is handled via Google Colab's built-in authentication for members of our team. We do not have the Google embeddings API publicly accessible, but our storage container is so the professor can see our results.

## 1. Install Dependencies & Authenticate
Authenticate with Google Cloud using Colab's built-in auth and retrieve an identity token for calling the Cloud Run endpoint.

In [ ]:
from google.colab import auth
auth.authenticate_user()

import subprocess
token = subprocess.check_output(
    ["gcloud", "auth", "print-identity-token"],
    text=True
).strip()

print("Authentication successful.")

## 2. Configuration
Set the input file, output file, Cloud Run function URL, and batch size used throughout the pipeline.

In [ ]:
import requests
import pandas as pd
import time
import os

INPUT_FILE = "phishtank_cleaned_minimal.txt"
OUTPUT_FILE = "local_embeddings_results.csv"
FUNCTION_URL = "https://embed-text-files-XXXXXXXXXXX.us-east1.run.app" # Redacted to prevent high costs on a shared notebook
BATCH_SIZE = 200 # The max the Gemini Embeddings API can handle in one request

# Below is the code of the embeddings cloud run function. Having it as a function rather than running locally lets it scale, authenticate, and cross communicate easily.
'''
# Needed for Cloud Run to recognize this as a web function to deploy
import functions_framework
from flask import jsonify, Request

# Google Gemini AI's client library
from google import genai

# Initialize client outside the function for better performance
genai_client = genai.Client(
    vertexai=True,
    project="datascience-489121",
    location="us-east1",
)

EMBEDDING_MODEL = "gemini-embedding-001" # The model from Google Gemini AI that lets us generate vector embeddings for text data

@functions_framework.http
def process_embeddings_worker(request: Request):
    req_json = request.get_json(silent=True)
    
    # Expecting: {"texts": ["string1", "string2", ...]}
    texts = req_json.get('texts', [])
    
    if not texts:
        return jsonify({"error": "No texts provided"}), 400

    try:
        # Call Vertex AI Embedding API
        result = genai_client.models.embed_content(
            model=EMBEDDING_MODEL,
            contents=texts,
        )
        
        # Format the response
        output = []
        for text, emb in zip(texts, result.embeddings):
            output.append({
                "text": text,
                "embedding": emb.values
            }) # As can be seen, the output is the embedded content as a JSON object with a vector list

        return jsonify({"results": output}), 200

    except Exception as e:
        return jsonify({"error": str(e)}), 500
        '''

## 3. Read Input Data
Load the input text file and count the total number of lines to be embedded.

In [ ]:
with open(INPUT_FILE, 'r') as f:
    lines = [line.strip() for line in f if line.strip()]

print(f"Total lines to process: {len(lines)}")

## 4. Process Batches of Inputs & Save Embeddings
Send lines to the Cloud Run embedding function in batches and append results to the output CSV incrementally.

In [ ]:
for i in range(0, len(lines), BATCH_SIZE):
    chunk = lines[i : i + BATCH_SIZE]

    payload = {"texts": chunk}

    try:
        headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json",
        }
        response = requests.post(FUNCTION_URL, json=payload, headers=headers)
        response.raise_for_status()
        data = response.json()

        df_batch = pd.DataFrame(data['results'])

        # Write header only on the first batch
        write_header = not os.path.exists(OUTPUT_FILE)
        df_batch.to_csv(OUTPUT_FILE, mode='a', index=False, header=write_header)

        print(f"Successfully processed batch {i // BATCH_SIZE + 1} ({i + len(chunk)}/{len(lines)})")

    except Exception as e:
        print(f"Error at batch starting at index {i}: {e}")
        time.sleep(2)

In [ ]:
if os.path.exists(OUTPUT_FILE):
    df = pd.read_csv(OUTPUT_FILE)
    print(f"Output saved to {OUTPUT_FILE} — {len(df)} rows, {len(df.columns)} columns")
    df.head()

You can see all of the larger embedded data at the below URLs:

[https://storage.googleapis.com/gscsteam1-colabresultbucket/](https://storage.googleapis.com/gscsteam1-colabresultbucket/)

[https://drive.google.com/drive/folders/1vULwJLjEuPXAWCR7eL5kq6g5KK3VU2wf?usp=sharing](https://drive.google.com/drive/folders/1vULwJLjEuPXAWCR7eL5kq6g5KK3VU2wf?usp=sharing)